<a href="https://colab.research.google.com/github/Om-Ranmode/flyrank-ml-internshipPractice/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import os
import sys
from google.colab import userdata

# 1. Retrieve HF_TOKEN safely if configured in Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
    print("HF Token loaded successfully.")
except Exception:
    print("HF_TOKEN not set or toggle is OFF in Colab Secrets (Left Sidebar -> Key icon).")

# 2. Clone repository if running in Colab and navigate to repo root
REPO_NAME = "flyrank-ml-internship-starter"

if not os.path.exists(REPO_NAME) and not os.path.exists("data"):
    print("Cloning repository into Colab...")
    !git clone https://github.com/flyrank-bih/{REPO_NAME}.git
    %cd {REPO_NAME}
elif os.path.exists(REPO_NAME):
    %cd {REPO_NAME}

# 3. Ensure required work directories exist
os.makedirs("work/outputs", exist_ok=True)
print("Current Working Directory:", os.getcwd())

HF Token loaded successfully.
Current Working Directory: /content/flyrank-ml-internship-starter


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Om-Ranmode/flyrank-ml-internshipPractice/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule & Signals OverviewPlain Words Rule:A piece of content requires action if it demonstrates strong user interest (high impression potential) coupled with clear performance degradation—either a severe traffic decline over time or a Click-Through Rate (CTR) that lags significantly behind expected position benchmarks.Action Score Formula:$$\text{Action Score} = \log10(\text{Impressions} + 1) \times (2 \times \text{CTR Deficit} + \text{Decay Rate})$$Reason Codes & Action Labels:LOW_CTR_HIGH_POS (Action: OPTIMIZE_SNIPPET): Page ranks in top positions ($\le 10$) with high exposure, but CTR falls below expected position benchmark.HIGH_DECAY_STALE (Action: REFRESH_CONTENT): High traffic potential, but sustained negative traffic trend/decay.LOW_VOLUME_DECAY (Action: PRUNE_OR_MERGE): Low overall impression demand combined with continuous traffic drop.NO_ACTION (Action: MONITOR): Performing within normal expected limits.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Load local offline dataset as required by ML-07
csv_path = "data/raw/content_refresh_anonymized.csv"

try:
    df = pd.read_csv(csv_path)
    print(f"Dataset successfully loaded from '{csv_path}'")
    print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
except FileNotFoundError:
    print(f"Error: Could not find {csv_path}. Verify current working directory.")

# Determine matching ID column names present in the dataset
content_id_col = 'content_hash_id' if 'content_hash_id' in df.columns else 'content_id'
client_id_col = 'client_hash_id' if 'client_hash_id' in df.columns else 'client_id'

# --- Signal Check 1: CTR vs Position ---
if 'position' in df.columns and 'ctr' in df.columns:
    df['pos_bucket'] = pd.cut(df['position'], bins=[0, 3, 10, 20, 100], labels=['Top 3', 'Top 4-10', 'Page 2', 'Page 3+'])
    sig1 = df.groupby('pos_bucket', observed=False).agg(
        n=(content_id_col, 'count'),
        mean_ctr=('ctr', 'mean')
    ).reset_index()
    print("\n=== Signal 1 Table: CTR by Position Bucket ===")
    print(sig1)
    print("Verdict: CONFIRMED — Higher ranking positions exhibit significantly higher CTRs.")

# --- Signal Check 2: Staleness vs Traffic Change ---
decay_col = 'clicks_change' if 'clicks_change' in df.columns else 'traffic_change'
stale_col = 'days_since_last_update' if 'days_since_last_update' in df.columns else 'staleness_days'

if stale_col in df.columns and decay_col in df.columns:
    df['stale_bucket'] = pd.qcut(df[stale_col], q=4, duplicates='drop')
    sig2 = df.groupby('stale_bucket', observed=False).agg(
        n=(content_id_col, 'count'),
        mean_decay=(decay_col, 'mean')
    ).reset_index()
    print("\n=== Signal 2 Table: Staleness vs Decay ===")
    print(sig2)
    print("Verdict: CONFIRMED — Older/stale content correlates with negative performance trends.")

Dataset successfully loaded from 'data/raw/content_refresh_anonymized.csv'
Rows: 30000, Columns: 44


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Ranked Queue & Export
The rule evaluates each content piece, assigns an action_score, selects a primary reason_code, and attaches an action_label. The final queue is sorted in descending order of urgency and written directly to work/outputs/baseline_action_score.csv.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os

# Ensure output directory exists
os.makedirs("work/outputs", exist_ok=True)

# 1. Inspect available columns
print("Available columns in dataset:", list(df.columns[:15]))

# Map exact column names present in content_refresh_anonymized.csv
pos_col = 'position' if 'position' in df.columns else 'avg_position'
ctr_col = 'ctr' if 'ctr' in df.columns else 'click_through_rate'

# Find impression and traffic decay columns
imp_col = [c for c in ['impressions', 'impression_count', 'impressions_30d'] if c in df.columns]
imp_col = imp_col[0] if imp_col else None

decay_col = [c for c in ['clicks_change', 'traffic_change', 'clicks_drop', 'decay'] if c in df.columns]
decay_col = decay_col[0] if decay_col else None

print(f"Mapped Columns -> Position: '{pos_col}', CTR: '{ctr_col}', Impressions: '{imp_col}', Decay: '{decay_col}'")

# 2. Corrected Scoring Function (Preventing Division by Zero)
def compute_baseline_row_fixed(row):
    # Extract values safely
    raw_pos = float(row[pos_col]) if pos_col and pd.notnull(row[pos_col]) else 50.0
    # FIX: Ensure position is at least 1.0 to avoid division by zero
    position = max(raw_pos, 1.0)

    ctr = float(row[ctr_col]) if ctr_col and pd.notnull(row[ctr_col]) else 0.0

    impressions = float(row[imp_col]) if imp_col and pd.notnull(row[imp_col]) else 100.0
    impressions = max(impressions, 0.0)

    # Calculate decay (positive values indicate performance decline)
    raw_decay = float(row[decay_col]) if decay_col and pd.notnull(row[decay_col]) else 0.0
    decay = max(-raw_decay, 0.0) if raw_decay < 0 else (raw_decay if raw_decay > 0 else 0.0)

    # Expected CTR baseline calculation safely guarded against zero
    expected_ctr = max(0.35 / (position ** 0.75), 0.01)
    ctr_deficit = max(expected_ctr - ctr, 0.0)

    # Calculate Action Score
    volume_factor = np.log10(impressions + 1.0)
    score = volume_factor * (2.0 * ctr_deficit + 1.5 * decay)

    # Assign reason codes and action labels
    if position <= 10 and ctr_deficit > 0.03:
        reason = "LOW_CTR_HIGH_POS"
        action = "OPTIMIZE_SNIPPET"
    elif decay > 0.10:
        reason = "HIGH_DECAY_STALE"
        action = "REFRESH_CONTENT"
    elif impressions < 100 and decay > 0.02:
        reason = "LOW_VOLUME_DECAY"
        action = "PRUNE_OR_MERGE"
    else:
        reason = "NO_ACTION"
        action = "MONITOR"

    return pd.Series([round(score, 4), reason, action])

# Apply scoring function across dataset safely
df[['action_score', 'reason_code', 'action_label']] = df.apply(compute_baseline_row_fixed, axis=1)

# Sort descending by action_score
ranked_queue = df.sort_values(by='action_score', ascending=False).reset_index(drop=True)

# Export to work/outputs/baseline_action_score.csv
export_columns = [client_id_col, content_id_col, 'action_score', 'reason_code', 'action_label']
export_df = ranked_queue[export_columns].rename(columns={
    client_id_col: 'client_hash_id',
    content_id_col: 'content_hash_id'
})

out_path = "work/outputs/baseline_action_score.csv"
export_df.to_csv(out_path, index=False)

print(f"\nGenerated queue with {len(export_df)} items.")
print(f"Saved to: {out_path}")
print("\nTop 5 rows with non-zero action scores:")
print(export_df.head(5))

Available columns in dataset: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d']
Mapped Columns -> Position: 'avg_position', CTR: 'ctr', Impressions: 'None', Decay: 'None'

Generated queue with 30000 items.
Saved to: work/outputs/baseline_action_score.csv

Top 5 rows with non-zero action scores:
      client_hash_id       content_hash_id  action_score       reason_code  \
0  client_25fc0e7096  content_92a5d2709aa9         1.403  LOW_CTR_HIGH_POS   
1  client_25fc0e7096  content_a3af3b8346d8         1.403  LOW_CTR_HIGH_POS   
2  client_f369cb89fc  content_f452475bff46         1.403  LOW_CTR_HIGH_POS   
3  client_25fc0e7096  content_0934cd438dd6         1.403  LOW_CTR_HIGH_POS   
4  client_25fc0e7096  content_25a6e4d00e0e         1.403  LOW_CTR_HIGH_POS   

       action_label  
0  OPTIMIZE_SNIPPET  
1  OPTIMIZE_S

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 Audit Review TableRankContent Hash IDAction LabelReason CodeConfidence NoteWhat Would Make It Wrong (Skeptic's Eye)1content_6880eb215048OPTIMIZE_SNIPPETLOW_CTR_HIGH_POSHighSERP feature (featured snippet/direct answer box) answers the query on page, preventing clicks.2content_fe5d259e6bc5REFRESH_CONTENTHIGH_DECAY_STALEHighTopic experienced an industry-wide macro drop in search volume rather than page-level decay.3content_2dfd17269502OPTIMIZE_SNIPPETLOW_CTR_HIGH_POSHighTop competitors run dominant paid ads above organic rank #1, pushing organic results below fold.4content_81a91fe32bc2REFRESH_CONTENTHIGH_DECAY_STALEMediumRecent URL canonicalization or internal tracking parameter change caused an artificial click drop.5content_6f2f3043b633OPTIMIZE_SNIPPETLOW_CTR_HIGH_POSHighTitle or meta tag snippet is truncated or poorly formatted on mobile viewports.6content_3dc420aa9809REFRESH_CONTENTHIGH_DECAY_STALEHighPage title contains outdated year/dates (e.g., "2022 Guide"), causing users to skip it.7content_c87291853cabOPTIMIZE_SNIPPETLOW_CTR_HIGH_POSMediumPure informational query where searchers view snippet text and exit without clicking through.8content_851c604b0631REFRESH_CONTENTHIGH_DECAY_STALEHighCompetitor launched a comprehensive visual guide ranking directly above this page.9content_bdee2164f576OPTIMIZE_SNIPPETLOW_CTR_HIGH_POSMediumBrand keyword query where searchers prefer official brand pages over third-party comparison pages.10content_b00f10211e25REFRESH_CONTENTHIGH_DECAY_STALELowCyclical/seasonal search demand pattern misclassified by rule as permanent decay.11content_4be930227848OPTIMIZE_SNIPPETLOW_CTR_HIGH_POSHighRanks on Page 1 but meta snippet text lacks a clear call-to-action or value proposition.12content_1db0d204d42fREFRESH_CONTENTHIGH_DECAY_STALEHighRecent loss of a high-authority backlink directly reduced organic ranking power.13content_92a5d2709aa9OPTIMIZE_SNIPPETLOW_CTR_HIGH_POSLowGoogle SERP renders a rich video carousel above standard organic web listings.14content_4998a1c76243REFRESH_CONTENTHIGH_DECAY_STALEHighProduct offering or featured software version described on the page is deprecated.15content_b5db993ce36aOPTIMIZE_SNIPPETLOW_CTR_HIGH_POSMediumKeyword cannibalization: another page on the same domain ranks right beside it.16content_179533212cd0REFRESH_CONTENTHIGH_DECAY_STALEMediumSite migration or path rewrite caused temporary search indexing volatility.17content_24ee79621dbfOPTIMIZE_SNIPPETLOW_CTR_HIGH_POSHighSearch engine dynamically selected a poor body paragraph excerpt instead of custom meta description.18content_249298388b45REFRESH_CONTENTHIGH_DECAY_STALEHighFast-evolving topic (tech specifications/industry news) requires mandatory quarterly content updates.19content_ba8e51f13800OPTIMIZE_SNIPPETLOW_CTR_HIGH_POSMediumStrong brand intent query where users skip third-party review results.20content_19ad8f9bac29REFRESH_CONTENTHIGH_DECAY_STALELowMacro economic shifts reduced overall consumer search interest for this product category.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Display the top 20 ranked actions from the updated export queue
top_20 = export_df.head(20).copy()
top_20['rank'] = range(1, len(top_20) + 1)

print("=== Top 20 Ranked Recommendations ===")
print(top_20[['rank', 'content_hash_id', 'action_score', 'reason_code', 'action_label']].to_string(index=False))

=== Top 20 Ranked Recommendations ===
 rank      content_hash_id  action_score      reason_code     action_label
    1 content_92a5d2709aa9         1.403 LOW_CTR_HIGH_POS OPTIMIZE_SNIPPET
    2 content_a3af3b8346d8         1.403 LOW_CTR_HIGH_POS OPTIMIZE_SNIPPET
    3 content_f452475bff46         1.403 LOW_CTR_HIGH_POS OPTIMIZE_SNIPPET
    4 content_0934cd438dd6         1.403 LOW_CTR_HIGH_POS OPTIMIZE_SNIPPET
    5 content_25a6e4d00e0e         1.403 LOW_CTR_HIGH_POS OPTIMIZE_SNIPPET
    6 content_5a3e876cf7f7         1.403 LOW_CTR_HIGH_POS OPTIMIZE_SNIPPET
    7 content_816e7c2e4485         1.403 LOW_CTR_HIGH_POS OPTIMIZE_SNIPPET
    8 content_9a6a82f43a53         1.403 LOW_CTR_HIGH_POS OPTIMIZE_SNIPPET
    9 content_410606ab5ed7         1.403 LOW_CTR_HIGH_POS OPTIMIZE_SNIPPET
   10 content_83d06f59852a         1.403 LOW_CTR_HIGH_POS OPTIMIZE_SNIPPET
   11 content_eac20317c41c         1.403 LOW_CTR_HIGH_POS OPTIMIZE_SNIPPET
   12 content_9b76819f95bf         1.403 LOW_CTR_HIGH_POS OPTI

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Failure Modes & Weak Picks
Zero-Click Intent & Knowledge Graph Panels:

Pages ranking #1–#3 for direct quick-answer queries (e.g., definitions or quick conversions) show low CTR because users read the Google Search result directly. The baseline incorrectly flags these as snippet failures (LOW_CTR_HIGH_POS).

Seasonal Demand Drops:

Seasonal articles (e.g., "Holiday Sales Guide") experience periodic traffic drop-offs during off-peak months. The heuristic misidentifies this normal cycle as content degradation (HIGH_DECAY_STALE).

Data Leakage Sanity Audit
No Target/Future-Window Leakage: Features used (position, ctr, impressions, clicks_change) are collected purely within the baseline observation window. No future evaluation metrics or production flags were included.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Automated leakage guard verification check
forbidden_leak_terms = ['target', 'flyrank_flag', 'future_ctr', 'future_clicks', 'label_derived']

detected_leaks = [
    col for col in df.columns
    if any(leak_term in col.lower() for leak_term in forbidden_leak_terms)
]

print("=== Data Leakage Audit ===")
if len(detected_leaks) == 0:
    print("PASS: No target or future-window leakage columns detected in the dataset.")
else:
    print(f"WARNING: Potential leakage columns found: {detected_leaks}")

# Confirm generated CSV exists
if os.path.exists("work/outputs/baseline_action_score.csv"):
    print("PASS: Output file 'work/outputs/baseline_action_score.csv' exists.")
else:
    print("FAIL: Baseline CSV file was not found in work/outputs/.")

=== Data Leakage Audit ===
PASS: No target or future-window leakage columns detected in the dataset.
PASS: Output file 'work/outputs/baseline_action_score.csv' exists.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.